In [ ]:
import os
import re
import json
import time
import random
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Set

import pandas as pd
import requests
from tqdm import tqdm
from dateutil import parser as date_parser


EXA_API_URL = "https://api.exa.ai/search"

os.environ["EXA_API_KEY"] = "..."

In [ ]:
def parse_dt_safe(dt_str: Optional[str]) -> Optional[datetime]:
    if dt_str is None or pd.isna(dt_str):
        return None

    try:
        dt = date_parser.parse(str(dt_str))

        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)

        return dt.astimezone(timezone.utc)

    except Exception:
        return None


def to_iso_z(dt: datetime) -> str:
    """
    Приводит datetime к формату:
        2024-01-03T14:11:38.915Z
    """
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)

    return dt.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def sample_prediction_date(
    start_date: str,
    end_date: str,
    rng: random.Random,
) -> Optional[datetime]:
    start_dt = parse_dt_safe(start_date)
    end_dt = parse_dt_safe(end_date)

    if start_dt is None or end_dt is None:
        return None

    if start_dt >= end_dt:
        return None

    total_seconds = int((end_dt - start_dt).total_seconds())

    if total_seconds <= 1:
        return None

    # Чтобы prediction_date не совпала с endDate
    offset_seconds = rng.randint(1, total_seconds - 1)

    return start_dt + pd.Timedelta(seconds=offset_seconds).to_pytimedelta()


def build_search_query(question: str) -> str:
    query = str(question).strip()

    if query.endswith("?"):
        query = query[:-1]

    if query.lower().startswith("will "):
        query = query[5:]

    return query

In [ ]:
def search_exa_news(
    question: str,
    prediction_date: datetime,
    num_results: int = 15,
    max_text_chars: int = 1200,
    sleep_sec: float = 0.25,
) -> List[Dict[str, Any]]:
    api_key = os.environ.get("EXA_API_KEY")

    if not api_key:
        raise RuntimeError("Не найден EXA_API_KEY.")

    cutoff_iso = to_iso_z(prediction_date)
    search_query = build_search_query(question)

    payload = {
        "query": search_query,
        "type": "auto",
        "category": "news",
        "numResults": num_results,
        "endPublishedDate": cutoff_iso,
        "contents": {
            "highlights": True,
            "text": {
                "maxCharacters": max_text_chars
            }
        }
    }

    headers = {
        "x-api-key": api_key,
        "Content-Type": "application/json",
    }

    response = requests.post(
        EXA_API_URL,
        headers=headers,
        json=payload,
        timeout=60,
    )

    if response.status_code != 200:
        error_text = response.text[:1000]

        if (
            response.status_code == 402
            or "NO_MORE_CREDITS" in error_text
            or "exceeded your credits limit" in error_text
            or "credits limit" in error_text
        ):
            raise RuntimeError(f"NO_MORE_CREDITS: {error_text}")

        raise RuntimeError(
            f"Exa API error {response.status_code}: {error_text}"
        )

    data = response.json()
    results = data.get("results", [])

    cleaned_results = []

    for item in results:
        cleaned_results.append(
            {
                "title": item.get("title"),
                "url": item.get("url"),
                "publishedDate": item.get("publishedDate"),
                "author": item.get("author"),
                "text": item.get("text"),
                "highlights": item.get("highlights"),
                "score": item.get("score"),
            }
        )

    time.sleep(sleep_sec)

    return cleaned_results

In [ ]:
def filter_news_before_prediction_date(
    news_context: List[Dict[str, Any]],
    prediction_date: datetime,
) -> List[Dict[str, Any]]:
    clean_news = []

    for item in news_context:
        pub_dt = parse_dt_safe(item.get("publishedDate"))

        if pub_dt is None:
            continue

        if pub_dt >= prediction_date:
            continue

        clean_news.append(item)

    return clean_news


def check_news_dates(
    news_context: List[Dict[str, Any]],
    prediction_date: datetime,
) -> bool:
    """
    Проверяет, что все новости строго раньше prediction_date.
    """
    if not news_context:
        return False

    for item in news_context:
        pub_dt = parse_dt_safe(item.get("publishedDate"))

        if pub_dt is None:
            return False

        if pub_dt >= prediction_date:
            return False

    return True

In [ ]:
def extract_entity_phrases(question: str) -> List[str]:
    question = str(question)
    entities = []

    # фразы в кавычках
    quoted = re.findall(r"[\"']([^\"']+)[\"']", question)

    for q in quoted:
        q = q.strip()
        if len(q) >= 3:
            entities.append(q.lower())

    capitalized_phrases = re.findall(
        r"\b(?:[A-Z][a-zA-Z0-9]+(?:\s+|$)){2,}",
        question
    )

    bad_phrases = {
        "Will",
        "Season",
        "Prime Video",
    }

    for phrase in capitalized_phrases:
        phrase = phrase.strip()
        phrase = re.sub(r"\s+", " ", phrase)

        if phrase in bad_phrases:
            continue

        if len(phrase) >= 5:
            entities.append(phrase.lower())

    known_entities = [
        "adult swim",
        "rick and morty",
        "joe vs carole",
        "joe exotic",
        "carole baskin",
        "peacock",
        "amazon prime video",
        "prime video",
        "netflix",
        "hbo max",
        "hulu",
        "disney",
        "disney+",
        "marvel",
        "dc",
        "warner bros",
        "paramount",
        "universal",
        "rotten tomatoes",
        "oscars",
        "grammys",
        "emmys",
        "golden globes",
        "epstein",
        "jimmy kimmel",
    ]

    q_lower = question.lower()

    for ent in known_entities:
        if ent in q_lower:
            entities.append(ent)

    seen = set()
    unique_entities = []

    for ent in entities:
        ent = ent.strip().lower()

        if ent and ent not in seen:
            seen.add(ent)
            unique_entities.append(ent)

    return unique_entities


def get_keywords_from_question(question: str) -> List[str]:
    stop_words = {
        "will", "the", "a", "an", "by", "on", "of", "and", "or", "to",
        "receive", "premiere", "release", "released", "first", "episode",
        "season", "limited", "series", "score", "higher", "before", "after",
        "with", "from", "that", "this", "into", "have", "has", "than",
        "over", "under", "less", "more", "least", "most", "between",
        "critics", "critic", "rotten", "tomatoes", "date", "guide",
        "video", "prime", "stream", "streaming", "watch", "official",
        "trailer", "teaser", "review", "reviews", "rating", "ratings",
        "guilty", "year", "record", "documents", "document", "name",
        "named", "say", "says", "make", "made",
    }

    raw_words = re.findall(r"[A-Za-z0-9']+", str(question).lower())

    keywords = []

    for word in raw_words:
        word = word.strip("'").lower()

        if len(word) < 4:
            continue

        if word in stop_words:
            continue

        keywords.append(word)

    seen = set()
    unique_keywords = []

    for word in keywords:
        if word not in seen:
            seen.add(word)
            unique_keywords.append(word)

    return unique_keywords


def strict_relevance_filter(
    news_context: List[Dict[str, Any]],
    question: str,
    min_keyword_matches: int = 2,
) -> List[Dict[str, Any]]:
    entities = extract_entity_phrases(question)
    keywords = get_keywords_from_question(question)

    clean_news = []

    for item in news_context:
        searchable_text = " ".join(
            [
                str(item.get("title") or ""),
                str(item.get("text") or ""),
                str(item.get("url") or ""),
            ]
        ).lower()

        entity_matches = [ent for ent in entities if ent in searchable_text]
        keyword_matches = [kw for kw in keywords if kw in searchable_text]

        if entities:
            if len(entity_matches) >= 1:
                item["relevance_entity_matches"] = entity_matches
                item["relevance_keyword_matches"] = len(keyword_matches)
                item["relevance_keywords_used"] = keywords
                item["relevance_entities_used"] = entities
                clean_news.append(item)
        else:
            if len(keyword_matches) >= min_keyword_matches:
                item["relevance_entity_matches"] = []
                item["relevance_keyword_matches"] = len(keyword_matches)
                item["relevance_keywords_used"] = keywords
                item["relevance_entities_used"] = entities
                clean_news.append(item)

    return clean_news

In [ ]:
def get_done_row_indices(output_csv_path: Path) -> Set[int]:
    if not output_csv_path.exists() or output_csv_path.stat().st_size == 0:
        return set()

    done_df = pd.read_csv(output_csv_path)

    if "row_index" not in done_df.columns:
        return set()

    return set(done_df["row_index"].dropna().astype(int).tolist())


def clean_credit_limit_rows(output_csv_path: Path) -> None:
    if not output_csv_path.exists() or output_csv_path.stat().st_size == 0:
        print("Output file does not exist or is empty.")
        return

    df = pd.read_csv(output_csv_path)

    if "news_error" not in df.columns:
        print("Column news_error not found.")
        return

    mask_bad = df["news_error"].fillna("").str.contains(
        "NO_MORE_CREDITS|exceeded your credits limit|credits limit",
        case=False,
        regex=True,
    )

    print("Удаляем строк с ошибкой лимита:", int(mask_bad.sum()))

    df_clean = df[~mask_bad].copy()
    df_clean.to_csv(output_csv_path, index=False, encoding="utf-8")

    print("Осталось строк:", len(df_clean))

In [ ]:
def enrich_csv_with_news_to_csv(
    input_csv_path: Path,
    output_csv_path: Path,
    limit: Optional[int] = None,
    seed: int = 42,
    raw_num_results: int = 15,
    final_num_results: int = 8,
    max_text_chars: int = 1200,
    min_keyword_matches: int = 2,
    resume: bool = True,
    save_every: int = 10,
) -> None:
    df = pd.read_csv(input_csv_path)

    done_row_indices = set()

    if resume:
        done_row_indices = get_done_row_indices(output_csv_path)
        print(f"Resume mode: found {len(done_row_indices)} already processed row_index values.")

    remaining_indices = [
        idx for idx in df.index
        if idx not in done_row_indices
    ]

    if limit is not None:
        remaining_indices = remaining_indices[:limit]

    print(f"Total input rows: {len(df)}")
    print(f"Rows remaining for this run: {len(remaining_indices)}")

    rng = random.Random(seed)

    processed_this_run = 0
    errors = 0
    rows_with_news = 0
    rows_bad_dates = 0
    rows_no_relevant_news = 0

    buffer = []

    def flush_buffer():
        nonlocal buffer

        if not buffer:
            return

        out_df = pd.DataFrame(buffer)

        file_exists = output_csv_path.exists()
        write_header = not file_exists or output_csv_path.stat().st_size == 0

        out_df.to_csv(
            output_csv_path,
            mode="a",
            index=False,
            header=write_header,
            encoding="utf-8",
        )

        buffer = []

    try:
        iterator = df.loc[remaining_indices].iterrows()

        for idx, row in tqdm(
            iterator,
            total=len(remaining_indices),
            desc="Adding news to CSV dataset",
        ):
            question = row.get("question")
            start_date = row.get("startDate")
            end_date = row.get("endDate")

            out = row.to_dict()

            out["row_index"] = int(idx)
            out["search_query"] = build_search_query(question) if pd.notna(question) else None

            out["prediction_date"] = None
            out["prediction_before_deadline"] = None

            out["raw_news_count"] = 0
            out["date_filtered_news_count"] = 0
            out["relevance_filtered_news_count"] = 0
            out["filtered_news_count"] = 0

            # в CSV списки/словари храним как JSON-строку
            out["news_context"] = "[]"
            out["news_dates_ok"] = False
            out["news_error"] = None

            start_dt = parse_dt_safe(start_date)
            end_dt = parse_dt_safe(end_date)

            if pd.isna(question) or start_dt is None or end_dt is None:
                out["news_error"] = "missing question/startDate/endDate"
                errors += 1

                buffer.append(out)
                processed_this_run += 1

                if len(buffer) >= save_every:
                    flush_buffer()

                continue

            if start_dt >= end_dt:
                out["news_error"] = "bad dates: startDate >= endDate"
                out["prediction_before_deadline"] = False
                rows_bad_dates += 1
                errors += 1

                buffer.append(out)
                processed_this_run += 1

                if len(buffer) >= save_every:
                    flush_buffer()

                continue

            prediction_dt = sample_prediction_date(
                start_date=start_date,
                end_date=end_date,
                rng=rng,
            )

            if prediction_dt is None:
                out["news_error"] = "could not sample prediction_date"
                errors += 1

                buffer.append(out)
                processed_this_run += 1

                if len(buffer) >= save_every:
                    flush_buffer()

                continue

            out["prediction_date"] = to_iso_z(prediction_dt)
            out["prediction_before_deadline"] = prediction_dt < end_dt

            try:
                raw_news = search_exa_news(
                    question=question,
                    prediction_date=prediction_dt,
                    num_results=raw_num_results,
                    max_text_chars=max_text_chars,
                )

                date_clean_news = filter_news_before_prediction_date(
                    news_context=raw_news,
                    prediction_date=prediction_dt,
                )

                relevant_news = strict_relevance_filter(
                    news_context=date_clean_news,
                    question=question,
                    min_keyword_matches=min_keyword_matches,
                )

                final_news = relevant_news[:final_num_results]

                out["raw_news_count"] = len(raw_news)
                out["date_filtered_news_count"] = len(date_clean_news)
                out["relevance_filtered_news_count"] = len(relevant_news)
                out["filtered_news_count"] = len(final_news)

                out["news_context"] = json.dumps(final_news, ensure_ascii=False)
                out["news_dates_ok"] = check_news_dates(final_news, prediction_dt)
                out["news_error"] = None

                if len(final_news) > 0:
                    rows_with_news += 1
                else:
                    rows_no_relevant_news += 1

            except Exception as e:
                error_msg = str(e)

                if (
                    "NO_MORE_CREDITS" in error_msg
                    or "exceeded your credits limit" in error_msg
                    or "credits limit" in error_msg
                ):
                    print("\nExa credits закончились. Останавливаю запуск без записи мусорной строки.")
                    break

                out["news_context"] = "[]"
                out["news_dates_ok"] = False
                out["news_error"] = error_msg
                errors += 1

            buffer.append(out)
            processed_this_run += 1

            if len(buffer) >= save_every:
                flush_buffer()

    finally:
        flush_buffer()

    print("Done.")
    print(f"Input rows: {len(df)}")
    print(f"Processed rows this run: {processed_this_run}")
    print(f"Rows with errors this run: {errors}")
    print(f"Rows with at least one clean relevant news item this run: {rows_with_news}")
    print(f"Rows with bad start/end dates this run: {rows_bad_dates}")
    print(f"Rows with zero relevant news after filters this run: {rows_no_relevant_news}")
    print(f"Output saved to: {output_csv_path}")

In [ ]:
def inspect_output_csv(output_csv_path: Path, max_rows: int = 10) -> None:
    df = pd.read_csv(output_csv_path)

    print("rows:", len(df))
    print("unique row_index:", df["row_index"].nunique() if "row_index" in df.columns else "no row_index")
    print("errors:", df["news_error"].notna().sum() if "news_error" in df.columns else "no news_error")
    print("rows with news:", (df["filtered_news_count"] > 0).sum() if "filtered_news_count" in df.columns else "no filtered_news_count")
    print("avg filtered news:", df["filtered_news_count"].mean() if "filtered_news_count" in df.columns else "no filtered_news_count")
    print("bad prediction dates:", (df["prediction_before_deadline"] == False).sum() if "prediction_before_deadline" in df.columns else "no prediction_before_deadline")

    for i, row in df.head(max_rows).iterrows():
        print("\n" + "=" * 80)
        print("ROW:", i)
        print("row_index:", row.get("row_index"))
        print("QUESTION:", row.get("question"))
        print("startDate:", row.get("startDate"))
        print("endDate:", row.get("endDate"))
        print("prediction_date:", row.get("prediction_date"))
        print("prediction_before_deadline:", row.get("prediction_before_deadline"))

        print("raw_news_count:", row.get("raw_news_count"))
        print("date_filtered_news_count:", row.get("date_filtered_news_count"))
        print("relevance_filtered_news_count:", row.get("relevance_filtered_news_count"))
        print("filtered_news_count:", row.get("filtered_news_count"))
        print("news_dates_ok:", row.get("news_dates_ok"))
        print("news_error:", row.get("news_error"))

        try:
            news = json.loads(row.get("news_context", "[]"))
        except Exception:
            news = []

        print("\nTop news:")
        for item in news[:3]:
            print("-", item.get("publishedDate"), "|", item.get("title"))
            print("  entity_matches:", item.get("relevance_entity_matches"))
            print("  keyword_matches:", item.get("relevance_keyword_matches"))
            print(" ", item.get("url"))


def show_progress(input_csv_path: Path, output_csv_path: Path) -> None:
    input_df = pd.read_csv(input_csv_path)

    if not output_csv_path.exists() or output_csv_path.stat().st_size == 0:
        print("Output пока пустой.")
        print("Всего строк во входном датасете:", len(input_df))
        print("Осталось:", len(input_df))
        return

    output_df = pd.read_csv(output_csv_path)

    done = output_df["row_index"].nunique() if "row_index" in output_df.columns else len(output_df)

    print("Всего строк во входном датасете:", len(input_df))
    print("Уже сохранено строк:", len(output_df))
    print("Уникальных row_index:", done)
    print("Осталось:", len(input_df) - done)

    if "news_error" in output_df.columns:
        print("Ошибок:", output_df["news_error"].notna().sum())

    if "filtered_news_count" in output_df.columns:
        print("Строк с новостями:", (output_df["filtered_news_count"] > 0).sum())
        print("Строк без новостей:", (output_df["filtered_news_count"] == 0).sum())

In [ ]:
input_path = Path("merged_culture_dataset_as_markets_format.csv")
output_path = Path("culture_markets_with_news_full.csv")

enrich_csv_with_news_to_csv(
    input_csv_path=input_path,
    output_csv_path=output_path,
    limit=None,
    seed=42,
    raw_num_results=15,
    final_num_results=8,
    max_text_chars=1200,
    min_keyword_matches=2,
    resume=True,
    save_every=10,
)

Resume mode: found 19072 already processed row_index values.
Total input rows: 19072
Rows remaining for this run: 0


Adding news to CSV dataset: 0it [00:00, ?it/s]

Done.
Input rows: 19072
Processed rows this run: 0
Rows with errors this run: 0
Rows with at least one clean relevant news item this run: 0
Rows with bad start/end dates this run: 0
Rows with zero relevant news after filters this run: 0
Output saved to: culture_markets_with_news_full.csv


In [ ]:
import pandas as pd
from pathlib import Path

output_path = Path("culture_markets_with_news_full.csv")
backup_path = Path("culture_markets_with_news_full_BACKUP_before_fix.csv")

df = pd.read_csv(output_path)
df.to_csv(backup_path, index=False, encoding="utf-8")

print("Backup saved to:", backup_path)
print("Rows in backup:", len(df))

Backup saved to: culture_markets_with_news_full_BACKUP_before_fix.csv
Rows in backup: 19072


In [ ]:
import pandas as pd
from pathlib import Path

output_path = Path("culture_markets_with_news_full.csv")

df = pd.read_csv(output_path)

news_error = df["news_error"].fillna("").astype(str)

mask_wrong_api = news_error.str.contains(
    "Invalid API key|Exa API error 401",
    case=False,
    regex=True,
)

print("Всего строк до чистки:", len(df))
print("Строк с неправильным API key:", mask_wrong_api.sum())

if mask_wrong_api.sum() > 0:
    print("Минимальный row_index с ошибкой:", df.loc[mask_wrong_api, "row_index"].min())
    print("Максимальный row_index с ошибкой:", df.loc[mask_wrong_api, "row_index"].max())

df_fixed = df[~mask_wrong_api].copy()

df_fixed.to_csv(output_path, index=False, encoding="utf-8")

print("Всего строк после чистки:", len(df_fixed))
print("Уникальных row_index после чистки:", df_fixed["row_index"].nunique())
print("Файл перезаписан:", output_path)

Всего строк до чистки: 19072
Строк с неправильным API key: 0
Всего строк после чистки: 19072
Уникальных row_index после чистки: 19072
Файл перезаписан: culture_markets_with_news_full.csv


In [ ]:
input_path = Path("culture_markets_2024_to_now_closed_resolved_yesno_FINAL.csv")
output_path = Path("culture_markets_with_news_full (1).csv")

input_df = pd.read_csv(input_path)
output_df = pd.read_csv(output_path)

done = output_df["row_index"].nunique()

print("Всего строк во входном датасете:", len(input_df))
print("Уже обработано уникальных row_index:", done)
print("Осталось перепарсить:", len(input_df) - done)

print("Ошибок осталось:", output_df["news_error"].notna().sum())
print(
    "Ошибок Invalid API key осталось:",
    output_df["news_error"].fillna("").astype(str).str.contains("Invalid API key|Exa API error 401", case=False, regex=True).sum()
)

Всего строк во входном датасете: 19072
Уже обработано уникальных row_index: 19072
Осталось перепарсить: 0
Ошибок осталось: 641
Ошибок Invalid API key осталось: 0


Сейчас будем пытаться допарсить новости там где их не нашло


In [ ]:
import pandas as pd
import numpy as np
import re
import json
from collections import Counter
from pathlib import Path

path = Path("culture_markets_with_news_full (1).csv")
df = pd.read_csv(path, low_memory=False)

for col in [
    "raw_news_count",
    "date_filtered_news_count",
    "relevance_filtered_news_count",
    "filtered_news_count",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

df["has_news"] = df["filtered_news_count"] > 0
df["has_error"] = df["news_error"].notna()

empty = df[~df["has_news"]].copy()
empty_no_error = empty[~empty["has_error"]].copy()

print("Всего строк:", len(df))
print("С новостями:", df["has_news"].sum())
print("Без новостей:", len(empty))
print("С ошибками:", df["has_error"].sum())
print("Без новостей, но без ошибок:", len(empty_no_error))

print("\nПричины среди пустых без ошибок:")
print("Exa ничего не вернул:", (empty_no_error["raw_news_count"] == 0).sum())
print(
    "Exa вернул, но всё вылетело по дате:",
    (
        (empty_no_error["raw_news_count"] > 0)
        & (empty_no_error["date_filtered_news_count"] == 0)
    ).sum()
)
print(
    "Новости были до prediction_date, но вылетели по релевантности:",
    (
        (empty_no_error["date_filtered_news_count"] > 0)
        & (empty_no_error["relevance_filtered_news_count"] == 0)
    ).sum()
)

Всего строк: 19072
С новостями: 12251
Без новостей: 6821
С ошибками: 641
Без новостей, но без ошибок: 6180

Причины среди пустых без ошибок:
Exa ничего не вернул: 1
Exa вернул, но всё вылетело по дате: 80
Новости были до prediction_date, но вылетели по релевантности: 6099


In [ ]:
patterns = {
    "elon_tweet_post_count": r"\b(elon|musk)\b.*\b(tweet|tweets|post|posts)\b|\b(tweet|tweets|post|posts)\b.*\b(elon|musk)\b",
    "social_media_post_count_general": r"\b(tweet|tweets|post|posts|instagram|truth social| on x\b)\b",
    "google_searches": r"\bgoogle searched|searched on google|google searches|searches\b",
    "spotify_appstore_charts": r"\bspotify|app store|itunes|billboard|charts?\b",
    "eurovision_country": r"\beurovision\b",
    "box_office_gross": r"\bgross|box office|opening weekend|domestic\b",
    "awards": r"\boscars?|grammys?|emmys?|golden globes?|academy awards?|best picture|best actor|best actress\b",
    "crypto_token_listing_price": r"\b(crypto|cryptocurrency|token|\$[A-Za-z0-9]+|coin|doge|ethereum|bitcoin|etf|binance|coinbase|listed|market cap|sotheby|nft|floor price)\b",
    "sports_esports_match": r"\b(defeat|ko|fight|fighter|hospitalized|go the distance|ufc|wwe|nba|nfl|super bowl|soccer|championship)\b",
    "release_premiere_launch": r"\b(release|premiere|launch|public release|come out|debut)\b",
    "person_say_attend": r"\b(say|mention|attend|appear|interview)\b",
}


def assign_question_types(question):
    q = str(question)
    types = []

    for name, pattern in patterns.items():
        if re.search(pattern, q, flags=re.IGNORECASE):
            types.append(name)

    return types if types else ["other"]


df["question_types"] = df["question"].apply(assign_question_types)
empty_no_error["question_types"] = empty_no_error["question"].apply(assign_question_types)

rows = []

all_types = sorted(set(t for types in df["question_types"] for t in types))

for t in all_types:
    mask_all = df["question_types"].apply(lambda xs: t in xs)
    mask_empty = empty_no_error["question_types"].apply(lambda xs: t in xs)

    total = int(mask_all.sum())
    empty_count = int(mask_empty.sum())

    rows.append({
        "question_type": t,
        "total_rows": total,
        "empty_no_error_rows": empty_count,
        "empty_rate_among_type": empty_count / total if total > 0 else np.nan,
    })

type_summary = (
    pd.DataFrame(rows)
    .sort_values("empty_no_error_rows", ascending=False)
)

type_summary

,question_type,total_rows,empty_no_error_rows,empty_rate_among_type
8,social_media_post_count_general,3885,3467,0.892407
3,elon_tweet_post_count,3560,3218,0.903933
5,other,7923,1431,0.180613
10,spotify_appstore_charts,1498,379,0.253004
6,person_say_attend,1769,288,0.162804
4,eurovision_country,232,202,0.870690
0,awards,2122,169,0.079642
7,release_premiere_launch,326,108,0.331288
1,box_office_gross,1025,98,0.095610
9,sports_esports_match,391,56,0.143223


In [ ]:
for t in type_summary["question_type"].head(12):
    subset = empty_no_error[
        empty_no_error["question_types"].apply(lambda xs: t in xs)
    ]

    print("\n" + "=" * 100)
    print("TYPE:", t)
    print("empty rows:", len(subset))

    for q in subset["question"].head(12):
        print("-", q)


TYPE: social_media_post_count_general
empty rows: 3467
- Will Nick Fuentes post on X by May 10?
- Will Elon tweet less than 60 times?
- Will Elon tweet 60-74 times?
- Will Elon tweet 75-89 times?
- Will Elon tweet 90-104 times?
- Will Elon tweet 105-119 times?
- Will Elon tweet 120-134 times?
- Will Elon tweet 135-149 times?
- Will Elon tweet 150 or more times?
- Will Elon post less than 40 times?
- Will Elon post 40-49 times?
- Will Elon post 50-59 times?

TYPE: elon_tweet_post_count
empty rows: 3218
- Will Elon tweet less than 60 times?
- Will Elon tweet 60-74 times?
- Will Elon tweet 75-89 times?
- Will Elon tweet 90-104 times?
- Will Elon tweet 105-119 times?
- Will Elon tweet 120-134 times?
- Will Elon tweet 135-149 times?
- Will Elon tweet 150 or more times?
- Will Elon post less than 40 times?
- Will Elon post 40-49 times?
- Will Elon post 50-59 times?
- Will Elon post 60-69 times?

TYPE: other
empty rows: 1431
- Will 2024 be the hottest year on record?
- Will America ban Zyn i